# Evaluación de Tavily como search_tool

**Planteamiento.** La empresa quiere un chatbot para consultas web. Evaluamos Tavily como herramienta de búsqueda de un agente LangGraph (`claude-haiku-4-5`) midiendo comportamiento, coste en tokens y relevancia, y probando mejoras.

**Estructura:** Paso 1 (comportamiento y tokens) · Paso 2 (anatomía de Tavily) · Paso 3 (mejoras) · Conclusiones.

In [1]:
import sys, json
sys.path.append("..")

import pandas as pd
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain_tavily import TavilySearch

from src.mario_mendoza_langgraph_websearchagent.Websearch.agent import (
    build_system_prompt, create_agent, get_token_usage, print_tool_messages, run_agent,
)
from src.mario_mendoza_langgraph_websearchagent.Websearch.token_utils import (
    count_tokens, estimate_cost, judge_grounding, make_grounding_judge,
    run_experiment, schema_tokens, show_llm_tools, tool_results_to_df,
)

## Paso 1: ¿el agente usa la tool cuando debe?

En vez de una tool de fecha, el system prompt inyecta la fecha actual y el cutoff del modelo, y explica cuándo buscar y cuándo no.

In [2]:
print(build_system_prompt())

You are an expert on LangChain and LangGraph and a helpful general assistant.
Today's date is 2026-11-02. Your training knowledge ends in July 2025.
Always answer in the same language as the user's question.

Web searches are expensive, so first decide whether you really need one.

Answer directly from your own knowledge, WITHOUT searching, when the question is about:
- Stable knowledge from before July 2025: concepts, definitions, history, science, math, and how established libraries and APIs work.
- Writing or explaining code, text or reasoning that needs no external data.
- Simple arithmetic (use the multiply tool for multiplications).

Use the search tool ONLY when:
- The question is about events, releases or versions after July 2025, or about data that changes over time (news, prices, weather, 'latest', 'current', 'today').
- You do not know the answer, or you are not confident your knowledge is up to date.

When you search: make ONE focused search with a short query. Search again

In [3]:
agent = create_agent()

PREGUNTAS_PRUEBA = {
    "no necesita búsqueda": "¿Qué es un StateGraph en LangGraph?",
    "necesita búsqueda": "¿Quién ganó el último Gran Premio de Fórmula 1?",
}

resultados = {}
for tipo, pregunta in PREGUNTAS_PRUEBA.items():
    print("#" * 70)
    print(tipo.upper(), "->", pregunta)
    resultados[tipo] = run_agent(agent, pregunta, tool_max_chars=300)

######################################################################
NO NECESITA BÚSQUEDA -> ¿Qué es un StateGraph en LangGraph?
================================ Human Message =================================

¿Qué es un StateGraph en LangGraph?
================================== Ai Message ==================================

Un **StateGraph** en LangGraph es la clase fundamental para construir grafos de flujo de trabajo que mantienen y transforman un estado compartido.

## Conceptos clave:

**¿Qué es?**
StateGraph es una estructura que define un grafo dirigido donde:
- Los **nodos** representan funciones o acciones que procesan el estado
- Los **bordes** representan transiciones entre nodos
- El **estado** es un objeto compartido que se pasa y se modifica a través del flujo

**Características principales:**

1. **Estado tipado**: Defines la estructura del estado usando TypedDict o Pydantic, asegurando que todos los nodos trabajen con datos consistentes

2. **Nodos**: Son funciones 

In [4]:
for tipo, res in resultados.items():
    usage = get_token_usage(res["messages"])
    print(f"{tipo}: búsquedas = {usage['search_calls']}")

no necesita búsqueda: búsquedas = 0
necesita búsqueda: búsquedas = 1


### Acceso a los ToolMessage

In [5]:
messages = resultados["necesita búsqueda"]["messages"]
tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

tm = tool_messages[0]
print("name:", tm.name)
print("tool_call_id:", tm.tool_call_id)
print("chars:", len(tm.content))
print(tm.content[:600])

name: tavily_search
tool_call_id: toolu_01PRe59QQsLVLxWxgmxzpx4W
chars: 6354
{"query": "último Gran Premio Fórmula 1 2026 ganador", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://depor.com/full-deportes/quien-gano-el-gran-premio-de-mexico-2026-en-vivo-hoy-resultados-de-la-carrera-f1-en-directo-y-como-quedo-kimi-antonelli-formula-uno-nnda-nnrt-noticia", "title": "¿Quién ganó el Gran Premio de México 2026 hoy?: Resultados de la carrera F1 en directo y cómo quedó Kimi Antonelli | FULL-DEPORTES | DEPOR", "content": "El gran protagonista del día fue Antonelli: con su tercer puesto a bordo del Mercedes, el joven italiano de 20 años ac


In [6]:
print_tool_messages(messages, max_chars=300)

TOOL MESSAGE 1: tavily_search (6354 chars)
Query:         último Gran Premio Fórmula 1 2026 ganador
Results:       5
Response time: 1.9 s

[1] ¿Quién ganó el Gran Premio de México 2026 hoy?: Resultados de la carrera F1 en directo y cómo quedó Kimi Antonelli | FULL-DEPORTES | DEPOR
    URL:     https://depor.com/full-deportes/quien-gano-el-gran-premio-de-mexico-2026-en-vivo-hoy-resultados-de-la-carrera-f1-en-directo-y-como-quedo-kimi-antonelli-formula-uno-nnda-nnrt-noticia
    Score:   0.8452414
    Content (1345 chars): El gran protagonista del día fue Antonelli: con su tercer puesto a bordo del Mercedes, el joven italiano de 20 años acumuló la puntuación requerida para sentenciar el campeonato a su favor, estableciendo un nuevo récord como el campeón del mundo. ## ¿Quién ganó el Gran Premio de México 2026 de Fórmu... [+1042 chars]

[2] Resultados de F1 2026 | Calendario de carreras, ganadores y clasificación del campeonato | TicketsGP
    URL:     https://www.tickets.gp/es/info/f1/res

### Métricas de token_usage

In [7]:
filas = []
for i, m in enumerate([m for m in messages if isinstance(m, AIMessage)], start=1):
    uso = m.usage_metadata
    filas.append({
        "llamada": i,
        "input_tokens": uso["input_tokens"],
        "output_tokens": uso["output_tokens"],
        "pidio_tool": len(m.tool_calls) > 0,
    })
display(pd.DataFrame(filas))

pd.Series(get_token_usage(messages))

,llamada,input_tokens,output_tokens,pidio_tool
0,1,2571,120,True
1,2,5114,282,False


llm_calls                      2.0
search_calls                   1.0
input_tokens                7685.0
output_tokens                402.0
total_tokens                8087.0
tool_result_tokens          2423.0
tool_input_tokens_billed    2423.0
tool_share_of_input           31.5
cache_read_tokens              0.0
cache_creation_tokens          0.0
dtype: float64

### Cómo se determinan los tokens de los ToolMessage (no de los tool_calls)

La API no devuelve tokens por mensaje, solo `usage_metadata` por llamada al LLM. El peso del ToolMessage se obtiene comparando dos llamadas consecutivas:

- **Llamada 1:** input = system prompt + pregunta + schema de las tools (`bind_tools()`). El modelo responde con un `AIMessage` que contiene los `tool_calls`. Esos tokens los genera el modelo, así que cuentan en `output_tokens` de la llamada 1.
- **Llamada 2:** input = todo lo anterior + el `AIMessage` con `tool_calls` + el `ToolMessage`.

Por tanto:

```
tool_result_tokens = input_2 - input_1 - output_1
```

Ejemplo numérico (ilustrativo): input_1 = 1250, output_1 = 38, input_2 = 3410, entonces el ToolMessage pesa 3410 - 1250 - 38 = 2122 tokens.

Diferencias clave:

- **tool_calls:** los genera el modelo (cuestan *output* una vez y luego *input* en las llamadas siguientes).
- **ToolMessage:** lo genera Tavily (solo cuesta *input*, pero se reenvía en cada llamada posterior). Por eso `get_token_usage` devuelve dos cifras: `tool_result_tokens` (contados una vez) y `tool_input_tokens_billed` (contados cada vez que se reenvían).

**Matiz con Claude:** `output_tokens` incluye los tokens de thinking, y `input_tokens` ya incluye los tokens de caché. Validamos el cálculo con un conteo independiente (API `count_tokens` de Anthropic, con unos pocos tokens de overhead de formato).

In [8]:
texto_tools = "".join(str(m.content) for m in tool_messages)
usage = get_token_usage(messages)
print("Por diferencia de input:", usage["tool_result_tokens"])
print("count_tokens API:       ", count_tokens(texto_tools))

Por diferencia de input: 2423
count_tokens API:        2418


In [9]:
# Coste fijo por llamada de los schemas de las tools (multiply + tavily_search)
print("Tokens del schema de tools por llamada:", schema_tokens())

Tokens del schema de tools por llamada: 2260


## Paso 2: entender el search_tool

### Estructura de lo que devuelve Tavily

In [10]:
datos = json.loads(tool_messages[0].content)
print("Claves de la respuesta:", list(datos.keys()))
print("Claves de cada resultado:", list(datos["results"][0].keys()))

tool_results_to_df(messages)

Claves de la respuesta: ['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']
Claves de cada resultado: ['url', 'title', 'content', 'score', 'raw_content', 'id']


,query,title,url,score,content_chars,content_tokens
0,último Gran Premio Fórmula 1 2026 ganador,¿Quién ganó el Gran Premio de México 2026 hoy?...,https://depor.com/full-deportes/quien-gano-el-...,0.845241,1345,413
1,último Gran Premio Fórmula 1 2026 ganador,Resultados de F1 2026 | Calendario de carreras...,https://www.tickets.gp/es/info/f1/resultados,0.843393,345,121
2,último Gran Premio Fórmula 1 2026 ganador,"Temporada 2026 de Fórmula 1 - Wikipedia, la en...",https://es.wikipedia.org/wiki/Temporada_2026_d...,0.788860,1275,476
3,último Gran Premio Fórmula 1 2026 ganador,Instagram,https://www.instagram.com/p/DfNzH4FFrM5,0.730463,492,171
4,último Gran Premio Fórmula 1 2026 ganador,Antonelli se corona campeón de Fórmula 1 en Mé...,https://soymotor.com/f1/noticias/cronica-carre...,0.722312,1271,497


### Parámetros disponibles y API que ve el LLM

In [11]:
schema_tavily = TavilySearch(max_results=5).args_schema.model_json_schema()
print(json.dumps(schema_tavily, indent=2, ensure_ascii=False)[:2500])

{
  "additionalProperties": true,
  "description": "Input for [TavilySearch]",
  "properties": {
    "query": {
      "description": "Search query to look up",
      "title": "Query",
      "type": "string"
    },
    "include_domains": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": [],
      "description": "A list of domains to restrict search results to.\n\n        Use this parameter when:\n        1. The user explicitly requests information from specific websites (e.g., \"Find climate data from nasa.gov\")\n        2. The user mentions an organization or company without specifying the domain (e.g., \"Find information about iPhones from Apple\")\n\n        In both cases, you should determine the appropriate domains (e.g., [\"nasa.gov\"] or [\"apple.com\"]) and set this parameter.\n\n        Results will ONLY come from the specified d

In [12]:
tools_llm = show_llm_tools()
texto = json.dumps(tools_llm, ensure_ascii=False)
print("Tokens del JSON de tools (count_tokens API):", count_tokens(texto))
print(json.dumps(tools_llm, indent=2, ensure_ascii=False)[:3000])

Tokens del JSON de tools (count_tokens API): 1767
[
  {
    "name": "multiply",
    "input_schema": {
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    },
    "description": "Multiply `a` and `b`.\n\n    Args:\n        a: First int\n        b: Second int"
  },
  {
    "name": "tavily_search",
    "input_schema": {
      "properties": {
        "query": {
          "description": "Search query to look up",
          "type": "string"
        },
        "include_domains": {
          "anyOf": [
            {
              "items": {
                "type": "string"
              },
              "type": "array"
            },
            {
              "type": "null"
            }
          ],
          "default": [],
          "description": "A list of domains to restrict search results to.\n\n        Use this parameter 

### Snippets y score: ¿son la parte relevante de la página?

Sustituye las preguntas por tus ejemplos (el de Alejandro y el de Miguel Ángel), abre cada URL y compara el snippet con la página real. Rellena `relevante_manual` con True/False.

In [13]:
buscador = TavilySearch(max_results=5)

PREGUNTAS_EJEMPLO = [
    "COMPLETA: pregunta del ejemplo de Alejandro",
    "COMPLETA: pregunta del ejemplo de Miguel Ángel",
]

filas = []
for pregunta in PREGUNTAS_EJEMPLO:
    respuesta = buscador.invoke({"query": pregunta})
    for r in respuesta["results"]:
        filas.append({
            "pregunta": pregunta,
            "titulo": r["title"][:50],
            "url": r["url"],
            "score": r["score"],
            "snippet": r["content"][:300],
            "relevante_manual": None,   # rellenar a mano
        })

df_snippets = pd.DataFrame(filas)
df_snippets

,pregunta,titulo,url,score,snippet,relevante_manual
0,COMPLETA: pregunta del ejemplo de Alejandro,[FREE] Completa el mensaje de Alejandro con el...,https://brainly.com/question/20276907,0.478408,"Dec 18, 2020 — Completa el mensaje de Alejandr...",None
1,COMPLETA: pregunta del ejemplo de Alejandro,"Alejandro Magno - Wikipedia, la enciclopedia l...",https://es.wikipedia.org/wiki/Alejandro_Magno,0.440590,"Durante los siglos XVI y XVII, fue habitual la...",None
2,COMPLETA: pregunta del ejemplo de Alejandro,Alejandro tenía 21 años y un diagnóstico que p...,https://www.facebook.com/lacorrientecom/videos...,0.097881,Alejandro tenía 21 años y un diagnóstico que p...,None
3,COMPLETA: pregunta del ejemplo de Alejandro,Alejandro Fernández RESPONDE a los rumores sob...,https://www.youtube.com/watch?v=stcyDWMEu1I,0.073437,Además de hacer frente a los rumores que dudan...,None
4,COMPLETA: pregunta del ejemplo de Alejandro,"José Alejandro Andalón, profesor de matemáticas",https://www.youtube.com/watch?v=G1Bwk7E0FJQ,0.050861,¿Puedes resolver este reto matemático? | José ...,None
5,COMPLETA: pregunta del ejemplo de Miguel Ángel,"Miguel Ángel - Wikipedia, la enciclopedia libre",https://es.wikipedia.org/wiki/Miguel_%C3%81ngel,0.387482,"Ángel su gran dominio del desnudo, junto con e...",None
6,COMPLETA: pregunta del ejemplo de Miguel Ángel,Michelangelo | Historia completa | Documental ...,https://www.youtube.com/watch?v=XPMmKoLXU0I,0.355686,propia visión. De niño fue enviado a Florencia...,None
7,COMPLETA: pregunta del ejemplo de Miguel Ángel,Miguel Ángel,https://linkedin.com/in/miguelangel,0.260783,# Miguel Ángel [...] ## Recommendations (0)\n-...,None
8,COMPLETA: pregunta del ejemplo de Miguel Ángel,Miguel Ángel: vida y obras del gran artista de...,https://www.youtube.com/watch?v=OueE0BuCeaM,0.253861,[15:16] republican government. The most import...,None
9,COMPLETA: pregunta del ejemplo de Miguel Ángel,Miguel Ángel,https://linkedin.com/in/miguelangelverdugo,0.185018,# Miguel Ángel [...] ## Recommendations (8)\n-...,None


### ¿Se corresponde el score con la relevancia?

Tras rellenar `relevante_manual`, compara los scores de los resultados relevantes y no relevantes. Si el score no separa bien ambos grupos, un umbral fijo no es buena idea. El cálculo interno del score es de Tavily: trátalo como caja negra y valídalo con tus etiquetas.

In [14]:
# Ejecutar después de rellenar relevante_manual
# df_snippets.groupby("relevante_manual")["score"].describe()

### ¿El modelo usa las respuestas o alucina?

In [15]:
juez = make_grounding_judge()
print(judge_grounding(messages, juez))

supported=True unsupported_claims=[]


## Paso 3: mejorar coste y relevancia

Cada configuración se pasa a `create_agent(...)`. Se comparan sobre las mismas preguntas (amplía a 15-20 para que sea representativo).

- `compact_results`: JSON compacto (sin espacios) y solo `title`, `url`, `content`, `score`.
- `max_results`: menos resultados.
- `min_score`: descarta resultados con score bajo. Si no queda ninguno, se devuelve un mensaje corto en lugar del JSON.
- `relevance_judge`: un LLM decide si cada resultado es relevante (esquema Pydantic con `relevant` y `explanation`).

In [16]:
PREGUNTAS = [
    "¿Quién ganó el último Gran Premio de Fórmula 1?",
    "¿Cuál es el precio actual del Bitcoin?",
    "¿Qué tiempo hará mañana en Madrid?",
    "¿Cuáles son las últimas novedades de LangGraph?",
    "¿Qué es un StateGraph en LangGraph?",
    "¿Quién es el presidente actual de Argentina?",
]

configuraciones = {
    "Línea base": {},
    "JSON compacto": {"compact_results": True},
    "3 resultados + compacto": {"max_results": 3, "compact_results": True},
    "score >= 0.5": {"min_score": 0.5, "compact_results": True},
    "Juez LLM de relevancia": {"relevance_judge": True, "compact_results": True},
    "Sin búsqueda (referencia)": {"use_search": False},
}

filas = [run_experiment(nombre, kwargs, PREGUNTAS, juez) for nombre, kwargs in configuraciones.items()]
df_resultados = pd.DataFrame(filas)
df_resultados

,config,searches,input_tokens,output_tokens,tool_result_tokens,tool_tokens_billed,agent_cost_usd,judge_cost_usd,total_cost_usd,answers_supported
0,Línea base,3,29882,2321,6466,6466,0.04149,0.00000,0.04149,3/3
1,JSON compacto,3,29322,2306,5906,5906,0.04085,0.00000,0.04085,2/3
2,3 resultados + compacto,3,27449,2215,4033,4033,0.03852,0.00000,0.03852,2/3
3,score >= 0.5,3,29322,2306,5906,5906,0.04085,0.00000,0.04085,2/3
4,Juez LLM de relevancia,3,28363,2272,4947,4947,0.03972,0.02621,0.06594,2/3
5,Sin búsqueda (referencia),0,4308,1801,0,0,0.01331,0.00000,0.01331,0/0


### Prompt caching

En Anthropic el caché es **explícito**: `create_agent(cache_system_prompt=True)` marca el system prompt con `cache_control`, y el caché cubre tools + system prompt. Hay un mínimo de tokens cacheables que depende del modelo (consúltalo en la documentación de prompt caching). Si el prefijo es más corto, `cache_creation_tokens` y `cache_read_tokens` saldrán en 0, y eso también es un hallazgo.

La primera llamada escribe el caché y las siguientes, dentro de 5 minutos, lo leen.

In [17]:
agente_cache = create_agent(cache_system_prompt=True)
pregunta = "¿Cuáles son las últimas novedades de LangGraph?"

for intento in (1, 2):
    res = agente_cache.invoke({"messages": [HumanMessage(content=pregunta)]})
    u = get_token_usage(res["messages"])
    print(f"Intento {intento}: input={u['input_tokens']}  cache_creation={u['cache_creation_tokens']}  "
          f"cache_read={u['cache_read_tokens']}  coste=${estimate_cost(u):.5f}")

Intento 1: input=7113  cache_creation=0  cache_read=0  coste=$0.00991
Intento 2: input=7114  cache_creation=0  cache_read=0  coste=$0.01011
